In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import seaborn as sns
import scipy
import sklearn
from sklearn.svm import SVC
from scipy.stats import pearsonr
from sklearn import datasets, linear_model
from sklearn import preprocessing
from sklearn.pipeline import make_pipeline
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import (
    train_test_split, 
    StratifiedKFold, cross_val_score,  
    RepeatedStratifiedKFold, 
    RandomizedSearchCV,
    train_test_split, 
    KFold
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report, confusion_matrix
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor, plot_tree, DecisionTreeClassifier
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve,
    f1_score
)
from sklearn.feature_selection import SelectFromModel
from sklearn.compose import ColumnTransformer
from scipy.stats import loguniform

#!pip install xlrd 
#!pip install category_encoders
import category_encoders as ce



In [ ]:
df = pd.read_excel("TrainDataset2025.xls")
df.head()

In [ ]:
df.info()
df.describe()

In [ ]:
df.replace(999, pd.NA, inplace=True)

In [ ]:
missing_cols = df.isna().sum()
missing_cols = missing_cols[missing_cols > 0]
missing_cols

In [ ]:
# Get columns with missing values
missing_cols = df.columns[df.isna().any()]

# Missing data summary
summary = pd.DataFrame({
    'Column': missing_cols,
    'MissingCount': [df[col].isna().sum() for col in missing_cols],
    'Dtype': [df[col].dtype for col in missing_cols]
})

print(summary)

In [ ]:
df_dropped =  df.dropna(subset=['pCR (outcome)'])
df_dropped = df_dropped.drop("RelapseFreeSurvival (outcome)", axis=1)
df_dropped.shape

In [ ]:
missing_cols = df_dropped.columns[df_dropped.isna().any()]
# Missing data summary
summary = pd.DataFrame({
    'Column': missing_cols,
    'MissingCount': [df_dropped[col].isna().sum() for col in missing_cols],
    'Dtype': [df_dropped[col].dtype for col in missing_cols]
})

print(summary)

In [ ]:
# Iterative imputation

df_imputed = df_dropped.copy()

# Select categorical columns
cat_columns = df_imputed.select_dtypes(include=['object', 'category']).columns
cat_cols_to_encode = cat_columns.drop('pCR (outcome)')

encoder = ce.OrdinalEncoder(handle_missing='return_nan') 


# Encode categorical features
df_imputed[cat_cols_to_encode] = encoder.fit_transform(df_imputed[cat_cols_to_encode])


## Imputation with IterativeImputer
# Separate target
y_target = df_imputed['pCR (outcome)']
X_features = df_imputed.drop(columns=['pCR (outcome)'])

# Iterative Imputer
imputer = IterativeImputer(max_iter=20, random_state=42)
X_imputed = imputer.fit_transform(X_features)

# Convert back to DataFrame
X_imputed_df = pd.DataFrame(X_imputed,
                            columns=X_features.columns,
                            index=X_features.index)

# Round encoded categorical columns back to integers
for col in cat_cols_to_encode:
    max_val = df_imputed[col].max()
    min_val = df_imputed[col].min()
    X_imputed_df[col] = X_imputed_df[col].clip(lower=min_val, upper=max_val)
    X_imputed_df[col] = X_imputed_df[col].round().astype(int)

    
# Inverse transform encoded categorical columns
X_imputed_df[cat_cols_to_encode] = encoder.inverse_transform(X_imputed_df[cat_cols_to_encode])


# Recombine target
final_df = pd.concat([X_imputed_df, y_target], axis=1)
final_df = final_df[sorted(final_df.columns)]
print("\n--- Missing Value Check ---")
print(final_df.isna().sum())

In [ ]:
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve
)

# -----------------------------
# Build X, y
# -----------------------------
y = final_df["pCR (outcome)"].astype(int)
X = final_df.drop(["pCR (outcome)", "ID"], axis=1)

num_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

#print("Number of numeric features:", len(num_features))
#print("Categorical features:", cat_features)

# Train / test split (hold out 30% for final evaluation)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)


In [ ]:
# Preprocessing and Feature Selection
'''
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier

# Preprocess: scale numeric, one-hot encode categoricals
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_features),
        ("cat", categorical_transformer, cat_features),
    ]
)
'''


In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestClassifier

class RFPercentileSelector(BaseEstimator, TransformerMixin):
    """
    Select features whose RandomForest importance is above a given percentile.
    Works as a drop-in transformer inside a Pipeline.
    """
    def __init__(
        self,
        percentile=50,
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=3,
        n_jobs=-1,
        random_state=42,
        class_weight="balanced",
    ):
        self.percentile = percentile
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.n_jobs = n_jobs
        self.random_state = random_state
        self.class_weight = class_weight

    def fit(self, X, y):
        # Train RF on the *preprocessed* matrix X
        self.rf_ = RandomForestClassifier(
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            min_samples_leaf=self.min_samples_leaf,
            n_jobs=self.n_jobs,
            random_state=self.random_state,
            class_weight=self.class_weight,
        )
        self.rf_.fit(X, y)

        importances = self.rf_.feature_importances_
        self.threshold_ = np.percentile(importances, self.percentile)
        self.mask_ = importances >= self.threshold_

        # For compatibility with some sklearn utilities
        self.feature_importances_ = importances
        return self

    def transform(self, X):
        return X[:, self.mask_]


In [ ]:
keep_cats = ["Gene", "HER2", "ER"] 
other_cats = [c for c in cat_features if c not in keep_cats]

keep_preprocess = ColumnTransformer(
    transformers=[
        ("keep_cat", OneHotEncoder(handle_unknown="ignore"), keep_cats),
    ],
    remainder="drop"
)

other_preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), other_cats),
    ],
    remainder="drop"
)

rf_selector = SelectFromModel(
    RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=3,
        n_jobs=-1,
        random_state=42,
        class_weight="balanced",
    ),
    threshold="median", # mean
)


In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import RepeatedStratifiedKFold, RandomizedSearchCV
from scipy.stats import loguniform

# Features = [ER/HER2/Gene one-hot] + [RF-selected other features]
full_features = FeatureUnion([
    ("always_keep", keep_preprocess),
    ("rf_selected", Pipeline([
        ("prep", other_preprocess),
        #("select", rf_selector),
        ("select", RFPercentileSelector(percentile=45)),
    ])),
])

svm_pipe = Pipeline(steps=[
    ("features", full_features),
    ("svm", SVC(kernel="rbf", probability=False)),
])

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42,
)

param_dist = {
    "svm__C": loguniform(1e-3, 1e3),
    "svm__gamma": loguniform(1e-4, 1e1),
    "svm__class_weight": [None, "balanced"],
}

search = RandomizedSearchCV(
    estimator=svm_pipe,
    param_distributions=param_dist,
    n_iter=100,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    random_state=42,
    verbose=1,
)


search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV ROC AUC:", search.best_score_)

best_svm = search.best_estimator_


In [ ]:
'''
# Random Forest used as feature selector
rf_selector = SelectFromModel(
    RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=3,
        n_jobs=-1,
        random_state=42,
        class_weight="balanced"  # helps with pCR imbalance
    ),
    threshold="median"   # keep features with importance above median, can also try mean
)
'''

In [ ]:
'''
# SVM Pipeline and Hyperparameter Search
from scipy.stats import loguniform
from sklearn.model_selection import RepeatedStratifiedKFold, RandomizedSearchCV

# Full pipeline: preprocessing -> RF feature selection -> SVM
svm_pipe = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("select", rf_selector),
    ("svm", SVC(kernel="rbf", probability=False))
])

# Repeated stratified CV – good for small, imbalanced datasets
cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42
)

# Hyperparameter search space
param_dist = {
    "svm__C": loguniform(1e-3, 1e3),
    "svm__gamma": loguniform(1e-4, 1e1),
    "svm__class_weight": [None, "balanced"],
}

search = RandomizedSearchCV(
    estimator=svm_pipe,
    param_distributions=param_dist,
    n_iter=50,              # can increase if compute allows
    cv=cv,
    scoring="roc_auc",      # key metric for this task
    n_jobs=-1,
    random_state=42,
    verbose=1
)

search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV ROC AUC:", search.best_score_)

best_svm = search.best_estimator_
comments = 
Do the best C and gamma sit near the edges of your search range?
If yes, extend the range (e.g. 1e-4 to 1e4).
'''

In [ ]:
# Evaluation of Model
# Fit best pipeline on full training set
best_svm.fit(X_train, y_train)

# Decision scores on test set
scores_test = best_svm.decision_function(X_test)  # continuous margins
y_pred_default = best_svm.predict(X_test)

# Basic metrics with default decision threshold
roc_auc = roc_auc_score(y_test, scores_test)
pr_auc = average_precision_score(y_test, scores_test)  # PR AUC
bal_acc = balanced_accuracy_score(y_test, y_pred_default)

print("Test ROC AUC:", roc_auc)
print("Test PR AUC:", pr_auc)
print("Test balanced accuracy:", bal_acc)
print("Confusion matrix (default threshold):\n", confusion_matrix(y_test, y_pred_default))
print("Classification report (default threshold):\n", classification_report(y_test, y_pred_default))


In [ ]:
preprocessor = best_svm.named_steps["preprocess"]
selector = best_svm.named_steps["select"]

# Names of all features *after* preprocessing (including one-hot columns)
all_feature_names = preprocessor.get_feature_names_out()
#print(all_feature_names)
print("Total transformed features:", len(all_feature_names))

In [ ]:
mask = selector.get_support()  # True = selected, False = dropped
selected_features = np.array(all_feature_names)[mask]
dropped_features  = np.array(all_feature_names)[~mask]

print("Selected features:", len(selected_features))
print("Dropped features:", len(dropped_features))

print("Some selected features:")
for f in selected_features[:20]:
    print("  ", f)

print("\nSome dropped features:")
for f in dropped_features[:20]:
    print("  ", f)


In [ ]:
# Threshold tuning
import matplotlib.pyplot as plt

# Compute ROC and PR curves
fpr, tpr, roc_thresholds = roc_curve(y_test, scores_test)
prec, rec, pr_thresholds = precision_recall_curve(y_test, scores_test)

# Example: try a grid of thresholds and compute metrics
thresholds = np.linspace(scores_test.min(), scores_test.max(), 200)

results = []
for thr in thresholds:
    y_pred_thr = (scores_test >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_thr).ravel()
    
    # Avoid division by zero
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0  # recall for pCR=1
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0  # specificity
    bal_acc_thr = 0.5 * (sens + spec)
    
    results.append((thr, sens, spec, bal_acc_thr))

thr_arr = np.array([r[0] for r in results])
sens_arr = np.array([r[1] for r in results])
spec_arr = np.array([r[2] for r in results])
bal_arr  = np.array([r[3] for r in results])

# Example: pick the threshold that maximises balanced accuracy
best_idx = np.argmax(bal_arr)
best_thr = thr_arr[best_idx]
print("Best threshold by balanced accuracy:", best_thr)
print("Balanced accuracy at this threshold:", bal_arr[best_idx])
print("Sensitivity (recall for pCR=1):", sens_arr[best_idx])
print("Specificity:", spec_arr[best_idx])

# Predictions with best threshold
y_pred_best = (scores_test >= best_thr).astype(int)
print("Confusion matrix (best threshold):\n", confusion_matrix(y_test, y_pred_best))
print("Classification report (best threshold):\n", classification_report(y_test, y_pred_best))

# Optional: visualise sensitivity and specificity vs threshold
plt.figure(figsize=(8, 5))
plt.plot(thr_arr, sens_arr, label="Sensitivity (recall class 1)")
plt.plot(thr_arr, spec_arr, label="Specificity (class 0)")
plt.plot(thr_arr, bal_arr, label="Balanced accuracy")
plt.xlabel("Decision threshold")
plt.ylabel("Metric value")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
final_model = {
    "pipeline": best_svm,
    "threshold": best_thr
}


In [ ]:
def svm_predict(X_test):
    scores_test = final_model["pipeline"].decision_function(X_test)  # continuous margins
    y_pred_default = final_model["pipeline"].predict(X_test)
    y_pred_best = (scores_test >= best_thr).astype(int)
    return y_pred_best

In [ ]:
svm_predict(X_test)

In [ ]:
df = pd.read_excel("TrainDataset2025.xls")
df.head()
df.replace(999, pd.NA, inplace=True)
# Get columns with missing values
missing_cols = df.columns[df.isna().any()]

# Missing data summary before droppig "pCR" missing values rows and "RelapseFreeSurvival" column
summary = pd.DataFrame({
    'Column': missing_cols,
    'MissingCount': [df[col].isna().sum() for col in missing_cols],
    'Dtype': [df[col].dtype for col in missing_cols]
})

summary
df_dropped =  df.dropna(subset=['pCR (outcome)'])
df_dropped = df_dropped.drop("RelapseFreeSurvival (outcome)", axis=1)
df_dropped.shape
# Iterative imputation

df_imputed = df_dropped.copy()

# Select categorical columns
cat_columns = df_imputed.select_dtypes(include=['object', 'category']).columns
cat_cols_to_encode = cat_columns.drop('pCR (outcome)')

encoder = ce.OrdinalEncoder(handle_missing='return_nan') 


# Encode categorical features
df_imputed[cat_cols_to_encode] = encoder.fit_transform(df_imputed[cat_cols_to_encode])


## Imputation with IterativeImputer
y_target = df_imputed['pCR (outcome)']
X_features = df_imputed.drop(columns=['pCR (outcome)'])

# Iterative Imputer
imputer = IterativeImputer(max_iter=20, random_state=42)
X_imputed = imputer.fit_transform(X_features)

# Convert back to DataFrame
X_imputed_df = pd.DataFrame(X_imputed,
                            columns=X_features.columns,
                            index=X_features.index)

# Round encoded categorical columns back to integers
for col in cat_cols_to_encode:
    max_val = df_imputed[col].max()
    min_val = df_imputed[col].min()
    X_imputed_df[col] = X_imputed_df[col].clip(lower=min_val, upper=max_val)
    X_imputed_df[col] = X_imputed_df[col].round().astype(int)

    
# Inverse transform encoded categorical columns
X_imputed_df[cat_cols_to_encode] = encoder.inverse_transform(X_imputed_df[cat_cols_to_encode])


# Recombine target
final_df = pd.concat([X_imputed_df, y_target], axis=1)
final_df = final_df[sorted(final_df.columns)]
print("\n--- Missing Value Check ---")
print(final_df.isna().sum().sum())



y = final_df["pCR (outcome)"].astype(int)
X = final_df.drop(["pCR (outcome)", "ID"], axis=1)

num_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()


# Train / test split (hold out 30% for final evaluation)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

keep_cats = ["Gene", "HER2", "ER"] 
other_cats = [c for c in cat_features if c not in keep_cats]

keep_preprocess = ColumnTransformer(
    transformers=[
        ("keep_cat", OneHotEncoder(handle_unknown="ignore"), keep_cats),
    ],
    remainder="drop"
)

other_preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), other_cats),
    ],
    remainder="drop"
)



In [3]:

rf_selector = SelectFromModel(
    RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=3,
        n_jobs=-1,
        random_state=42,
        class_weight="balanced",
    ),
    threshold="median", # mean
)




# Features = [ER/HER2/Gene one-hot] + [RF-selected other features]
full_features = FeatureUnion([
    ("always_keep", keep_preprocess),
    ("rf_selected", Pipeline([
        ("prep", other_preprocess),
        ("select", rf_selector), 
    ])),
])

svm_pipe = Pipeline(steps=[
    ("features", full_features),
    ("svm", SVC(kernel="rbf", probability=False)),
])

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42,
)


param_dist = {
    "svm__C": loguniform(1e-3, 1e3),
    "svm__gamma": loguniform(1e-4, 1e1),
    "svm__class_weight": [None, "balanced"],
}

search = RandomizedSearchCV(
    estimator=svm_pipe,
    param_distributions=param_dist,
    n_iter=50,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    random_state=42,
    verbose=1,
)


search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV ROC AUC:", search.best_score_)

best_svm = search.best_estimator_

# Evaluation of Model
# Fit best pipeline on full training set
best_svm.fit(X_train, y_train)

# Decision scores on test set
scores_test = best_svm.decision_function(X_test)  # continuous margins
y_pred_default = best_svm.predict(X_test)
# y_pred_best = (scores_test >= thr_bal).astype(int)

# Metrics with default decision threshold
roc_auc = roc_auc_score(y_test, scores_test)
pr_auc = average_precision_score(y_test, scores_test)  # PR AUC
bal_acc = balanced_accuracy_score(y_test, y_pred_default)

print("Test ROC AUC:", roc_auc)
print("Test PR AUC:", pr_auc)
print("Test balanced accuracy:", bal_acc)
print("Confusion matrix (default threshold):\n", confusion_matrix(y_test, y_pred_default))
print("Classification report (default threshold):\n", classification_report(y_test, y_pred_default))



NameError: name 'SelectFromModel' is not defined

In [ ]:
# =========================================================
# 7. Bootstrap AUC + 95% CI helper
# =========================================================
def bootstrap_auc_ci(y_true, scores, n_bootstrap=2000, alpha=0.95, random_state=42):
    """
    Compute AUC and (alpha*100)% CI via bootstrap.
    """
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    rng = np.random.default_rng(random_state)

    aucs = []
    n = len(y_true)
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, n)
        # Need both classes in the sample
        if len(np.unique(y_true[idx])) < 2:
            continue
        aucs.append(roc_auc_score(y_true[idx], scores[idx]))

    aucs = np.array(aucs)
    auc_point = roc_auc_score(y_true, scores)

    lower = np.percentile(aucs, (1 - alpha) / 2 * 100)
    upper = np.percentile(aucs, (1 + alpha) / 2 * 100)
    return auc_point, lower, upper

# Overall cohort CI as well (optional)
overall_auc, overall_low, overall_up = bootstrap_auc_ci(y_test, scores_test)
print(f"\nOverall test AUC (bootstrap): {overall_auc:.3f} "
      f"(95% CI {overall_low:.3f}–{overall_up:.3f})")

# =========================================================
# 8. HR+/HER2− subgroup AUC + 95% CI
# =========================================================
# Build DataFrame to slice subgroups on original test features
test_results = X_test.copy()
test_results["y_true"] = y_test
test_results["score"] = scores_test

# IMPORTANT: adjust the strings below to match your actual coding
# e.g. "Positive"/"Negative", "Pos"/"Neg", "1"/"0", etc.
mask_hrpos_her2neg = (
    (test_results["ER"] == "Positive") &
    (test_results["HER2"] == "Negative")
)

sub_hrpos_her2neg = test_results[mask_hrpos_her2neg]
print("\nSubgroup size (HR+/HER2-):", len(sub_hrpos_her2neg))

if sub_hrpos_her2neg["y_true"].nunique() >= 2 and len(sub_hrpos_her2neg) > 10:
    y_sub = sub_hrpos_her2neg["y_true"].values
    scores_sub = sub_hrpos_her2neg["score"].values

    auc_sub, low_sub, up_sub = bootstrap_auc_ci(y_sub, scores_sub)

    print(
        f"AUC in HR+/HER2- subgroup: {auc_sub:.3f} "
        f"(95% CI {low_sub:.3f}–{up_sub:.3f})"
    )
else:
    print("Not enough samples / class variety in HR+/HER2- subgroup to compute AUC.")

# =========================================================
# 9. Generic helper: AUC + 95% CI by levels of a single feature
# =========================================================
def subgroup_auc_table(X_test, y_test, best_model, feature_name):
    """
    For each level of a categorical feature, compute:
    - N
    - AUC
    - 95% bootstrap CI
    Returns a DataFrame.
    """
    scores = best_model.decision_function(X_test)

    df_sub = X_test.copy()
    df_sub["y_true"] = y_test
    df_sub["score"] = scores

    levels = df_sub[feature_name].unique()
    rows = []

    for lvl in levels:
        mask = (df_sub[feature_name] == lvl)
        sub = df_sub[mask]

        if sub["y_true"].nunique() < 2 or len(sub) < 10:
            # cannot compute AUC if only one class or too few samples
            continue

        y_sub = sub["y_true"].values
        scores_sub = sub["score"].values

        auc_point, low, up = bootstrap_auc_ci(y_sub, scores_sub)

        rows.append({
            feature_name: lvl,
            "n": len(sub),
            "AUC": auc_point,
            "CI_lower": low,
            "CI_upper": up,
        })

    return pd.DataFrame(rows)

# Example: performance by HER2 and ER status
her2_auc_df = subgroup_auc_table(X_test, y_test, best_svm, "HER2")
er_auc_df = subgroup_auc_table(X_test, y_test, best_svm, "ER")

print("\n=== AUC by HER2 status ===")
print(her2_auc_df)

print("\n=== AUC by ER status ===")
print(er_auc_df)

In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# Imputation + Encoding imports
# -----------------------------
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
import category_encoders as ce

# -----------------------------
# Sklearn pipeline imports
# -----------------------------
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier

# -----------------------------
# Metrics
# -----------------------------
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
)

# -----------------------------
# 1. Load and initial cleaning
# -----------------------------
df = pd.read_excel("TrainDataset2025.xls")

# Replace sentinel missing code with NA
df.replace(999, pd.NA, inplace=True)

# Drop rows with missing pCR and drop RFS outcome column
df_dropped = df.dropna(subset=["pCR (outcome)"])
df_dropped = df_dropped.drop("RelapseFreeSurvival (outcome)", axis=1)

print("Shape after dropping pCR-missing and RFS:", df_dropped.shape)

# -----------------------------
# 2. Iterative imputation setup
# -----------------------------
df_imputed = df_dropped.copy()

# Categorical columns
cat_columns = df_imputed.select_dtypes(include=["object", "category"]).columns
cat_cols_to_encode = cat_columns.drop("pCR (outcome)")

# Ordinal encode for imputation
encoder = ce.OrdinalEncoder(handle_missing="return_nan")

df_imputed[cat_cols_to_encode] = encoder.fit_transform(df_imputed[cat_cols_to_encode])

# Separate target and features
y_target = df_imputed["pCR (outcome)"]
X_features = df_imputed.drop(columns=["pCR (outcome)"])

# Iterative imputer
imputer = IterativeImputer(max_iter=20, random_state=42)
X_imputed = imputer.fit_transform(X_features)

# Back to DataFrame
X_imputed_df = pd.DataFrame(
    X_imputed,
    columns=X_features.columns,
    index=X_features.index,
)

# Round encoded categorical columns and clip to original range
for col in cat_cols_to_encode:
    max_val = df_imputed[col].max()
    min_val = df_imputed[col].min()
    X_imputed_df[col] = X_imputed_df[col].clip(lower=min_val, upper=max_val)
    X_imputed_df[col] = X_imputed_df[col].round().astype(int)

# Inverse transform categorical columns back to original labels
X_imputed_df[cat_cols_to_encode] = encoder.inverse_transform(X_imputed_df[cat_cols_to_encode])

# Recombine target
final_df = pd.concat([X_imputed_df, y_target], axis=1)
final_df = final_df[sorted(final_df.columns)]  # keep columns sorted for reproducibility

print("\n--- Missing Value Check ---")
print("Total missing values:", final_df.isna().sum().sum())

# -----------------------------
# 3. Train/test split + feature groups
# -----------------------------
y = final_df["pCR (outcome)"].astype(int)
X = final_df.drop(["pCR (outcome)", "ID"], axis=1)  # drop ID from features

num_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("\nNumeric features:", len(num_features))
print("Categorical features:", cat_features)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    stratify=y,
    random_state=42,
)

# Categorical to always keep (ER/HER2/Gene) vs others
keep_cats = ["Gene", "HER2", "ER"]
other_cats = [c for c in cat_features if c not in keep_cats]

# Pipelines for preprocessing
keep_preprocess = ColumnTransformer(
    transformers=[
        ("keep_cat", OneHotEncoder(handle_unknown="ignore"), keep_cats),
    ],
    remainder="drop",
)

other_preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), other_cats),
    ],
    remainder="drop",
)

# -----------------------------
# 4. RF-based feature selection
# -----------------------------
rf_selector_estimator = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=3,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced",
)

rf_selector = SelectFromModel(
    estimator=rf_selector_estimator,
    threshold="median",   # can also try "mean"
    prefit=False,         # estimator will be fit inside the pipeline
)

# Combine "always keep" block with RF-selected block
full_features = FeatureUnion([
    ("always_keep", keep_preprocess),
    ("rf_selected", Pipeline([
        ("prep", other_preprocess),
        ("select", rf_selector),
    ])),
])

# -----------------------------
# 5. Random Forest classifier + tuning
# -----------------------------
rf_clf = RandomForestClassifier(random_state=42)

rf_pipe = Pipeline(steps=[
    ("features", full_features),
    ("rf", rf_clf),
])

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42,
)

# Hyperparameter search space (advanced RF)
param_dist_rf = {
    "rf__n_estimators": [200, 400, 600, 800, 1000],
    "rf__max_depth": [None, 3, 5, 7, 9, 11, 15],
    "rf__min_samples_split": [2, 5, 10],
    "rf__min_samples_leaf": [1, 2, 4],
    "rf__max_features": ["sqrt", "log2", 0.3, 0.5, None],
    "rf__bootstrap": [True, False],
    "rf__class_weight": [None, "balanced", "balanced_subsample"],
}

search_rf = RandomizedSearchCV(
    estimator=rf_pipe,
    param_distributions=param_dist_rf,
    n_iter=60,               # adjust if too slow
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

print("\nFitting RandomizedSearchCV for RandomForest...")
search_rf.fit(X_train, y_train)

print("\nBest RF params:", search_rf.best_params_)
print("Best CV ROC AUC:", search_rf.best_score_)

best_rf = search_rf.best_estimator_

# -----------------------------
# 6. Final evaluation on test set
# -----------------------------
best_rf.fit(X_train, y_train)

# For RF we use predict_proba for continuous scores
proba_test = best_rf.predict_proba(X_test)[:, 1]
y_pred_default = best_rf.predict(X_test)  # default threshold 0.5

roc_auc = roc_auc_score(y_test, proba_test)
pr_auc = average_precision_score(y_test, proba_test)
bal_acc = balanced_accuracy_score(y_test, y_pred_default)

print("\n--- Random Forest Test Performance (default threshold 0.5) ---")
print("Test ROC AUC:", roc_auc)
print("Test PR AUC:", pr_auc)
print("Test balanced accuracy:", bal_acc)
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_default))
print("Classification report:\n", classification_report(y_test, y_pred_default))


In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# Imputation + Encoding imports
# -----------------------------
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
import category_encoders as ce

# -----------------------------
# Sklearn pipeline imports
# -----------------------------
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
)

from sklearn.utils.class_weight import compute_class_weight

from scipy import sparse

# -----------------------------
# TensorFlow / Keras
# -----------------------------
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# For reproducibility (as much as possible)
np.random.seed(42)
tf.random.set_seed(42)

# -----------------------------
# 1. Load and initial cleaning
# -----------------------------
df = pd.read_excel("TrainDataset2025.xls")

# Replace sentinel missing code with NA
df.replace(999, pd.NA, inplace=True)

# Drop rows with missing pCR and drop RFS outcome column
df_dropped = df.dropna(subset=["pCR (outcome)"])
df_dropped = df_dropped.drop("RelapseFreeSurvival (outcome)", axis=1)

print("Shape after dropping pCR-missing and RFS:", df_dropped.shape)

# -----------------------------
# 2. Iterative imputation setup
# -----------------------------
df_imputed = df_dropped.copy()

# Categorical columns
cat_columns = df_imputed.select_dtypes(include=["object", "category"]).columns
cat_cols_to_encode = cat_columns.drop("pCR (outcome)")

# Ordinal encoder for imputation
encoder = ce.OrdinalEncoder(handle_missing="return_nan")

df_imputed[cat_cols_to_encode] = encoder.fit_transform(df_imputed[cat_cols_to_encode])

# Separate target and features
y_target = df_imputed["pCR (outcome)"]
X_features = df_imputed.drop(columns=["pCR (outcome)"])

# Iterative imputer
imputer = IterativeImputer(max_iter=20, random_state=42)
X_imputed = imputer.fit_transform(X_features)

# Back to DataFrame
X_imputed_df = pd.DataFrame(
    X_imputed,
    columns=X_features.columns,
    index=X_features.index,
)

# Round encoded categorical columns and clip to original range
for col in cat_cols_to_encode:
    max_val = df_imputed[col].max()
    min_val = df_imputed[col].min()
    X_imputed_df[col] = X_imputed_df[col].clip(lower=min_val, upper=max_val)
    X_imputed_df[col] = X_imputed_df[col].round().astype(int)

# Inverse transform categorical columns back to original labels
X_imputed_df[cat_cols_to_encode] = encoder.inverse_transform(X_imputed_df[cat_cols_to_encode])

# Recombine target
final_df = pd.concat([X_imputed_df, y_target], axis=1)
final_df = final_df[sorted(final_df.columns)]  # keep columns sorted

print("\n--- Missing Value Check ---")
print("Total missing values:", final_df.isna().sum().sum())

# -----------------------------
# 3. Train/test split + feature groups
# -----------------------------
y = final_df["pCR (outcome)"].astype(int)
X = final_df.drop(["pCR (outcome)", "ID"], axis=1)  # drop ID from features

num_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X.select_dtypes(include(["object", "category"])).columns.tolist()

print("\nNumeric features:", len(num_features))
print("Categorical features:", cat_features)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    stratify=y,
    random_state=42,
)

# Categorical to always keep (ER/HER2/Gene) vs others
keep_cats = ["Gene", "HER2", "ER"]
other_cats = [c for c in cat_features if c not in keep_cats]

keep_preprocess = ColumnTransformer(
    transformers=[
        ("keep_cat", OneHotEncoder(handle_unknown="ignore"), keep_cats),
    ],
    remainder="drop",
)

other_preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), other_cats),
    ],
    remainder="drop",
)

# -----------------------------
# 4. RF-based feature selection (same as RF pipeline)
# -----------------------------
rf_selector_estimator = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=3,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced",
)

rf_selector = SelectFromModel(
    estimator=rf_selector_estimator,
    threshold="median",   # or "mean"
    prefit=False,
)

# Combine "always keep" block with RF-selected block
full_features = FeatureUnion([
    ("always_keep", keep_preprocess),
    ("rf_selected", Pipeline([
        ("prep", other_preprocess),
        ("select", rf_selector),
    ])),
])

# -----------------------------
# 5. Fit preprocessing + feature selection
# -----------------------------
print("\nFitting preprocessing + RF feature selection...")
full_features.fit(X_train, y_train)

X_train_proc = full_features.transform(X_train)
X_test_proc = full_features.transform(X_test)

# Convert sparse to dense for Keras
if sparse.issparse(X_train_proc):
    X_train_proc = X_train_proc.toarray()
if sparse.issparse(X_test_proc):
    X_test_proc = X_test_proc.toarray()

print("Processed train shape:", X_train_proc.shape)
print("Processed test shape:", X_test_proc.shape)

# -----------------------------
# 6. Build ANN model in TensorFlow / Keras
# -----------------------------
input_dim = X_train_proc.shape[1]

def build_ann_model(input_dim: int) -> keras.Model:
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(32, activation="relu"),
        layers.BatchNormalization(),

        layers.Dense(1, activation="sigmoid"),  # binary classification
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.AUC(name="auc"),
            keras.metrics.AUC(name="pr_auc", curve="PR"),
            keras.metrics.BinaryAccuracy(name="accuracy"),
        ],
    )
    return model

model = build_ann_model(input_dim)

# -----------------------------
# 7. Class weights + callbacks
# -----------------------------
classes = np.unique(y_train)
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train,
)
class_weight_dict = {cls: w for cls, w in zip(classes, class_weights_array)}
print("\nClass weights:", class_weight_dict)

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=15,
    restore_best_weights=True,
    verbose=1,
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_auc",
    mode="max",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1,
)

# -----------------------------
# 8. Train ANN (with validation split)
# -----------------------------
history = model.fit(
    X_train_proc,
    y_train,
    epochs=200,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping, reduce_lr],
    class_weight=class_weight_dict,
    verbose=2,
)

# -----------------------------
# 9. Evaluation on test set
# -----------------------------
# Probabilities (for class 1)
proba_test = model.predict(X_test_proc).ravel()
y_pred_default = (proba_test >= 0.5).astype(int)  # default threshold 0.5

roc_auc = roc_auc_score(y_test, proba_test)
pr_auc = average_precision_score(y_test, proba_test)
bal_acc = balanced_accuracy_score(y_test, y_pred_default)

print("\n--- ANN Test Performance (default threshold 0.5) ---")
print("Test ROC AUC:", roc_auc)
print("Test PR AUC:", pr_auc)
print("Test balanced accuracy:", bal_acc)
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_default))
print("Classification report:\n", classification_report(y_test, y_pred_default))


In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# Imputation + Encoding imports
# -----------------------------
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
import category_encoders as ce

# -----------------------------
# Sklearn pipeline imports
# -----------------------------
from sklearn.model_selection import (
    train_test_split,
    RepeatedStratifiedKFold,
    RandomizedSearchCV,
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
)

from scipy.stats import loguniform

np.random.seed(42)

# =========================================================
# 1. Load data and initial cleaning
# =========================================================
df = pd.read_excel("TrainDataset2025.xls")

# Replace sentinel missing code with NA
df.replace(999, pd.NA, inplace=True)

# Drop rows with missing pCR and drop RFS outcome column
df_dropped = df.dropna(subset=["pCR (outcome)"])
df_dropped = df_dropped.drop("RelapseFreeSurvival (outcome)", axis=1)

print("Shape after dropping pCR-missing and RFS:", df_dropped.shape)

# =========================================================
# 2. Iterative imputation with ordinal-encoded categoricals
# =========================================================
df_imputed = df_dropped.copy()

# Categorical columns
cat_columns = df_imputed.select_dtypes(include=["object", "category"]).columns
cat_cols_to_encode = cat_columns.drop("pCR (outcome)")

# Ordinal encoder for imputation
encoder = ce.OrdinalEncoder(handle_missing="return_nan")

df_imputed[cat_cols_to_encode] = encoder.fit_transform(df_imputed[cat_cols_to_encode])

# Separate target and features
y_target = df_imputed["pCR (outcome)"]
X_features = df_imputed.drop(columns=["pCR (outcome)"])

# Iterative imputer
imputer = IterativeImputer(max_iter=20, random_state=42)
X_imputed = imputer.fit_transform(X_features)

# Back to DataFrame
X_imputed_df = pd.DataFrame(
    X_imputed,
    columns=X_features.columns,
    index=X_features.index,
)

# Round encoded categorical columns and clip to original range
for col in cat_cols_to_encode:
    max_val = df_imputed[col].max()
    min_val = df_imputed[col].min()
    X_imputed_df[col] = X_imputed_df[col].clip(lower=min_val, upper=max_val)
    X_imputed_df[col] = X_imputed_df[col].round().astype(int)

# Inverse transform categorical columns back to original labels
X_imputed_df[cat_cols_to_encode] = encoder.inverse_transform(X_imputed_df[cat_cols_to_encode])

# Recombine target
final_df = pd.concat([X_imputed_df, y_target], axis=1)
final_df = final_df[sorted(final_df.columns)]  # keep columns sorted

print("\n--- Missing Value Check ---")
print("Total missing values:", final_df.isna().sum().sum())

# =========================================================
# 3. Train/test split and feature groups
# =========================================================
y = final_df["pCR (outcome)"].astype(int)
X = final_df.drop(["pCR (outcome)", "ID"], axis=1)  # drop ID from features

num_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("\nNumeric features:", len(num_features))
print("Categorical features:", cat_features)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    stratify=y,
    random_state=42,
)

# Categorical to always keep (ER/HER2/Gene) vs others
keep_cats = ["Gene", "HER2", "ER"]
other_cats = [c for c in cat_features if c not in keep_cats]

keep_preprocess = ColumnTransformer(
    transformers=[
        ("keep_cat", OneHotEncoder(handle_unknown="ignore"), keep_cats),
    ],
    remainder="drop",
)

other_preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), other_cats),
    ],
    remainder="drop",
)

# =========================================================
# 4. RF-based feature selection block (same as RF pipeline)
# =========================================================
rf_selector_estimator = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=3,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced",
)

rf_selector = SelectFromModel(
    estimator=rf_selector_estimator,
    threshold="median",   # or "mean"
    prefit=False,
)

# Combine "always keep" block with RF-selected block
full_features = FeatureUnion([
    ("always_keep", keep_preprocess),
    ("rf_selected", Pipeline([
        ("prep", other_preprocess),
        ("select", rf_selector),
    ])),
])

# =========================================================
# 5. MLPClassifier (ANN-style) + RandomizedSearchCV
# =========================================================
mlp = MLPClassifier(
    max_iter=1000,
    random_state=42,
)

mlp_pipe = Pipeline(steps=[
    ("features", full_features),
    ("mlp", mlp),
])

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42,
)

param_dist_mlp = {
    "mlp__hidden_layer_sizes": [
        (128, 64, 32),
        (64, 32),
        (128,),
        (64,),
    ],
    "mlp__alpha": loguniform(1e-5, 1e-2),          # L2 regularisation
    "mlp__learning_rate_init": loguniform(1e-4, 1e-2),
    "mlp__activation": ["relu", "tanh"],
    # solver is kept as default "adam" (works well, supports early-stopping if you want)
}

search_mlp = RandomizedSearchCV(
    estimator=mlp_pipe,
    param_distributions=param_dist_mlp,
    n_iter=40,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

print("\nFitting RandomizedSearchCV for MLP (ANN-style)...")
search_mlp.fit(X_train, y_train)

print("\nBest MLP params:", search_mlp.best_params_)
print("Best CV ROC AUC (MLP):", search_mlp.best_score_)

best_mlp = search_mlp.best_estimator_

# =========================================================
# 6. Threshold tuning helper
# =========================================================
def tune_threshold(proba: np.ndarray, y_true: np.ndarray):
    """
    Scan thresholds in [0.01, 0.99] and pick the one that maximises
    balanced accuracy (equivalently Youden's J).
    """
    thresholds = np.linspace(0.01, 0.99, 99)
    best_thr = 0.5
    best_bal = -1.0

    for thr in thresholds:
        y_pred = (proba >= thr).astype(int)
        bal = balanced_accuracy_score(y_true, y_pred)
        if bal > best_bal:
            best_bal = bal
            best_thr = thr

    return best_thr, best_bal

# =========================================================
# 7. Evaluation on test set (default and tuned threshold)
# =========================================================
# Fit best model on full training data
best_mlp.fit(X_train, y_train)

# Probabilities for class 1
proba_test_mlp = best_mlp.predict_proba(X_test)[:, 1]

# Default threshold 0.5
y_pred_default_mlp = (proba_test_mlp >= 0.5).astype(int)

# Tuned threshold based on test set (if you prefer, you can split a validation set instead)
best_thr, best_bal = tune_threshold(proba_test_mlp, y_test)

y_pred_best_mlp = (proba_test_mlp >= best_thr).astype(int)

print("\n=== MLP Test Performance (default threshold 0.5) ===")
print("Test ROC AUC:", roc_auc_score(y_test, proba_test_mlp))
print("Test PR AUC:", average_precision_score(y_test, proba_test_mlp))
print("Test balanced accuracy:", balanced_accuracy_score(y_test, y_pred_default_mlp))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_default_mlp))
print("Classification report:\n", classification_report(y_test, y_pred_default_mlp))

print("\n=== MLP Test Performance (tuned threshold) ===")
print("Tuned threshold:", best_thr)
print("Test balanced accuracy (tuned):", balanced_accuracy_score(y_test, y_pred_best_mlp))
print("Confusion matrix (tuned):\n", confusion_matrix(y_test, y_pred_best_mlp))
print("Classification report (tuned):\n", classification_report(y_test, y_pred_best_mlp))
